In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd 
import config
import firms_clean as fc

In [2]:
raw=fc.load_raw_firms()
raw.head()

Loaded 439 files -> 5,672,637 rows
Country is not inferred from the acquisition-area filename.


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type,source_file
0,10.72426,107.64754,342.41,0.76,0.77,2020-01-01,512,N,VIIRS,n,2,290.88,9.05,D,0,VIIRS_SNPP_SP_92.2_5.6_109.5_28.5_2020-01-01_5...
1,10.72939,107.64829,335.12,0.76,0.77,2020-01-01,512,N,VIIRS,n,2,290.80,7.56,D,0,VIIRS_SNPP_SP_92.2_5.6_109.5_28.5_2020-01-01_5...
2,10.91045,107.67505,343.38,0.75,0.77,2020-01-01,512,N,VIIRS,n,2,292.90,9.39,D,0,VIIRS_SNPP_SP_92.2_5.6_109.5_28.5_2020-01-01_5...
3,10.91296,107.67291,348.75,0.75,0.77,2020-01-01,512,N,VIIRS,n,2,293.16,8.66,D,0,VIIRS_SNPP_SP_92.2_5.6_109.5_28.5_2020-01-01_5...
4,11.98278,107.91705,338.62,0.69,0.74,2020-01-01,512,N,VIIRS,n,2,291.51,13.29,D,0,VIIRS_SNPP_SP_92.2_5.6_109.5_28.5_2020-01-01_5...


In [3]:
timed=fc.standardize_time(raw)
timed[["acq_date","acq_time","acq_datetime_utc","date","year","month"]].head()

Dates: 2020-01-01 to 2025-12-31
Years: [np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]


,acq_date,acq_time,acq_datetime_utc,date,year,month
0,2020-01-01,512,2020-01-01 05:12:00+00:00,2020-01-01,2020,1
1,2020-01-01,512,2020-01-01 05:12:00+00:00,2020-01-01,2020,1
2,2020-01-01,512,2020-01-01 05:12:00+00:00,2020-01-01,2020,1
3,2020-01-01,512,2020-01-01 05:12:00+00:00,2020-01-01,2020,1
4,2020-01-01,512,2020-01-01 05:12:00+00:00,2020-01-01,2020,1


In [4]:
fc.count_duplicates_by_key(timed)

                 definition  rows_removed  percent
           coordinates only           276      0.0
         coordinates + date             4      0.0
   coordinates + exact time             4      0.0
coordinates + time + sensor             4      0.0

A looser key removes more rows. But two satellites seeing the
same fire are two independent observations, not a duplicate.


,definition,rows_removed,percent
0,coordinates only,276,0.0
1,coordinates + date,4,0.0
2,coordinates + exact time,4,0.0
3,coordinates + time + sensor,4,0.0


In [5]:
deduped,n_removed=fc.drop_duplicates_firms(timed)
if "satellite" in deduped.columns:
    print()
    print(deduped.groupby(["instrument", "satellite"]).size().to_frame("rows"))

Duplicates removed: 4 of 5,672,637 (0.00%)

                         rows
instrument satellite         
VIIRS      N          5672633


In [6]:
normed=fc.normalize_confidence(deduped)

Confidence levels:
confidence_level
nominal    4808535
low         619022
high        245076

confidence_level    high     low  nominal
instrument                               
VIIRS             245076  619022  4808535


In [7]:
valid,drop_report=fc.filter_invalid(normed)

Invalid rows removed: 0 of 5,672,633
  (no rule fired - the raw data passed every check)


In [8]:
borders=fc.load_country_borders()
with_country=fc.assign_country(valid,borders=borders)
with_country[["latitude", "longitude", "country"]].head(10)

Country borders loaded: 242 countries
Country found for 5,641,344 of 5,672,633 hotspots (31,289 are in the sea or outside all borders)


,latitude,longitude,country
0,10.72426,107.64754,VNM
1,10.72939,107.64829,VNM
2,10.91045,107.67505,VNM
3,10.91296,107.67291,VNM
4,11.98278,107.91705,VNM
5,11.98403,107.92309,VNM
6,12.77694,106.82845,KHM
7,12.77918,106.82506,KHM
8,12.79443,106.84042,KHM
9,12.94421,106.81570,KHM


In [9]:
in_study,removed_report=fc.keep_study_countries(with_country)

Kept 5,030,113 of 5,672,633 hotspots in ['KHM', 'LAO', 'MMR', 'THA', 'VNM']
                          reason  rows_removed
in the sea / outside all borders         31289
              in another country        611231

Removed hotspots by country (top 5):
country
CHN    306668
IND    268958
BGD     31500
MYS      4092
IDN        13


In [10]:
with_bbox=fc.assign_country_bbox(in_study)
summary,mistakes=fc.compare_country_methods(with_bbox)
display(summary)
display(mistakes)

,comparable rows,box agrees with borders,accuracy %
0,5030113,2796947,55.6


,real country,box says,rows
6,THA,MMR,601847
4,LAO,THA,545098
1,KHM,THA,442991
2,KHM,VNM,361091
7,VNM,LAO,115410
3,LAO,MMR,76097
8,VNM,THA,45509
0,KHM,LAO,45015
5,THA,LAO,108


In [11]:
clean,report=fc.clean_firms(raw)

Dates: 2020-01-01 to 2025-12-31
Years: [np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
Duplicates removed: 4 of 5,672,637 (0.00%)
Invalid rows removed: 0 of 5,672,633
  (no rule fired - the raw data passed every check)
Confidence levels:
confidence_level
nominal    4808535
low         619022
high        245076

confidence_level    high     low  nominal
instrument                               
VIIRS             245076  619022  4808535
Country borders loaded: 242 countries
Country found for 5,641,344 of 5,672,633 hotspots (31,289 are in the sea or outside all borders)
Kept 5,030,113 of 5,672,633 hotspots in ['KHM', 'LAO', 'MMR', 'THA', 'VNM']
                          reason  rows_removed
in the sea / outside all borders         31289
              in another country        611231

Removed hotspots by country (top 5):
country
CHN    306668
IND    268958
BGD     31500
MYS      4092
IDN        13

  rows_in                5672637
  duplicat

In [12]:
path=fc.save_clean(clean)
reloaded=pd.read_parquet(path)
print(f"{len(reloaded):,} dòng, {len(reloaded.columns)} cột")
print(f"date dtype: {reloaded['date'].dtype}")

Saved 5,030,113 rows -> D:\Duanduhoche.Huy\wildfire-risk-analysis\data\processed\firms_clean.parquet
5,030,113 dòng, 25 cột
date dtype: datetime64[ns]
